# 客户价值与留存决策（模拟分析）

本节通过假设客户案例理解流失风险、客户价值、留存干预效果及成本之间的关系。

以下参数仅用于演示决策逻辑，并非 IBM 数据集中的实际利润、预测结果或干预效果。

## 1. 场景一：初始假设

本节仅使用两名模拟客户（A / B），不依赖 IBM 真实客户 CSV。情景中的月度利润、流失概率、成功挽留概率与联系成本均为**教学假设**，不是估计或预测结果。

**简化公式：** `预期净收益 = 流失概率 × 成功挽留概率 × 挽留后的贡献价值 − 联系成本`。

此处“成功挽留概率”表示干预对原本会流失客户产生的挽回效果。模拟采用挽留后持续 `6` 个月的简化假设。

In [1]:
import pandas as pd

demo_customers = pd.DataFrame({
    "customer_id": ["A", "B"],
    "churn_prob": [0.80, 0.50],
    "monthly_profit": [20, 200],
    "rescue_rate": [0.25, 0.25],
    "contact_cost": [10, 10]
})

demo_customers

,customer_id,churn_prob,monthly_profit,rescue_rate,contact_cost
0,A,0.8,20,0.25,10
1,B,0.5,200,0.25,10


In [2]:
months_saved = 6

demo_customers["value_if_saved"] = (
    demo_customers["monthly_profit"] * months_saved
)

demo_customers

,customer_id,churn_prob,monthly_profit,rescue_rate,contact_cost,value_if_saved
0,A,0.8,20,0.25,10,120
1,B,0.5,200,0.25,10,1200


In [3]:
demo_customers["expected_net_benefit"] = (
    demo_customers["churn_prob"]
    * demo_customers["rescue_rate"]
    * demo_customers["value_if_saved"]
    - demo_customers["contact_cost"]
).round(2)

demo_customers

,customer_id,churn_prob,monthly_profit,rescue_rate,contact_cost,value_if_saved,expected_net_benefit
0,A,0.8,20,0.25,10,120,14.0
1,B,0.5,200,0.25,10,1200,140.0


In [4]:
demo_customers.sort_values(
    "expected_net_benefit",
    ascending=False
)

,customer_id,churn_prob,monthly_profit,rescue_rate,contact_cost,value_if_saved,expected_net_benefit
1,B,0.5,200,0.25,10,1200,140.0
0,A,0.8,20,0.25,10,120,14.0


## 2. 场景二：降低客户 B 的挽留成功率

保留其他条件，仅把客户 B 的 `rescue_rate` 从 0.25 改为 0.01。修改输入后必须**重新运行收益公式**，旧的 `expected_net_benefit` 不会自动更新。

In [5]:
demo_customers.loc[
    demo_customers["customer_id"] == "B",
    "rescue_rate"
] = 0.01

demo_customers[["customer_id", "rescue_rate"]]

,customer_id,rescue_rate
0,A,0.25
1,B,0.01


In [6]:
# 重新计算预期净收益
demo_customers["expected_net_benefit"] = (
    demo_customers["churn_prob"]
    * demo_customers["rescue_rate"]
    * demo_customers["value_if_saved"]
    - demo_customers["contact_cost"]
).round(2)

# 按预期收益从大到小排序
demo_customers[
    ["customer_id", "rescue_rate", "expected_net_benefit"]
].sort_values(
    "expected_net_benefit",
    ascending=False
)

,customer_id,rescue_rate,expected_net_benefit
0,A,0.25,14.0
1,B,0.01,-4.0


## 3. 分析结论与局限性

- 场景一：客户 A 的预期净收益为 14，客户 B 为 140；在这些假设下 B 排序靠前。
- 场景二：将 B 的 `rescue_rate` 改为 0.01 并重新计算后，A 为 14，B 为 -4；在这些假设下 A 排序靠前。
- 本示例用于理解留存决策，不表示公司实际收益，也不能证明留存干预有真实效果。
- 后续独立建模时，需先评估客户流失预测模型，再讨论客户价值、干预增量效果和成本的估计方式。